# Median Per-Cell QC by Platform and Segmentation

Load AnnData tables embedded in MerXen `latest_spatialdata.zarr` outputs, compute per-cell transcript and detected-gene counts from each table's `X` matrix, remove low-count cells, and plot paired MERSCOPE/Xenium medians.

In [ ]:
from pathlib import Path

import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import spatialdata as sd
from IPython.display import display
from scipy import sparse
from tqdm.auto import tqdm

plt.rcParams.update({
    "figure.dpi": 140,
    "savefig.dpi": 300,
    "font.size": 10,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

In [ ]:
def find_repo_root() -> Path:
    """Find the MerXen repository from either repo root or notebooks/."""
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "README.md").exists() and (candidate / "results").exists():
            return candidate
    raise FileNotFoundError("Could not find a repo root containing README.md and results/.")


REPO_ROOT = find_repo_root()
RESULTS_DIR = REPO_ROOT / "results"
SUMMARY_FIGURE_PATH = REPO_ROOT / "notebooks" / "median_cell_qc_by_platform_segmentation.png"
DISTRIBUTION_PATH = REPO_ROOT / "notebooks" / "median_cell_qc_by_platform_segmentation_cell_distributions.csv.gz"
VIOLIN_FIGURE_PATH = REPO_ROOT / "notebooks" / "median_cell_qc_by_platform_segmentation_distributions.png"
FIGURE_PATH = SUMMARY_FIGURE_PATH

# Leave as None to use every result. Set to e.g. ["P5011"] while iterating.
SAMPLE_IDS = None

# Cells with 0 or 1 detected genes or transcripts are removed before medians.
MIN_COUNT_EXCLUSIVE = 1

# Keep cells whose 2D polygon area falls inside this micron-squared range.
MIN_CELL_AREA_UM2 = 5.0
MAX_CELL_AREA_UM2 = 400.0

# Keep the violin rendering responsive while saving the full filtered distribution.
VIOLIN_MAX_CELLS_PER_GROUP = 100_000
VIOLIN_RANDOM_SEED = 0

# Replace sample IDs with anonymous letter labels (A, B, C, …) in violin plots.
DEIDENTIFY_SAMPLE_LABELS = False

REPO_ROOT, RESULTS_DIR

In [ ]:
SEGMENTATION_STYLES = {
    "original_seg": {
        "table_key": "table_original",
        "label": "Out of instrument segmentation",
        "color": "black",
        "offset": -0.035,
    },
    "reseg": {
        "table_key": "table_MOSAIK_proseg",
        "label": "Optimised resegmentation",
        "color": "#d62728",
        "offset": 0.035,
    },
}

PLATFORM_X = {"merscope": 0.0, "xenium": 1.0}
PLATFORM_LABELS = {"merscope": "MERSCOPE", "xenium": "Xenium"}


def discover_spatialdata_outputs(results_dir: Path) -> pd.DataFrame:
    """Find latest SpatialData zarrs without loading their arrays."""
    rows = []
    for zarr_path in sorted(results_dir.glob("*/*/latest/latest_spatialdata.zarr")):
        pair_id, platform = zarr_path.relative_to(results_dir).parts[:2]
        if platform not in PLATFORM_X:
            continue
        if SAMPLE_IDS is not None and pair_id not in SAMPLE_IDS:
            continue
        rows.append({"pair_id": pair_id, "platform": platform, "zarr_path": zarr_path})

    if not rows:
        raise FileNotFoundError(f"No latest_spatialdata.zarr outputs found below {results_dir}.")
    return pd.DataFrame(rows)


spatialdata_outputs = discover_spatialdata_outputs(RESULTS_DIR)
display(spatialdata_outputs.assign(zarr_path=lambda df: df["zarr_path"].map(lambda p: p.relative_to(REPO_ROOT))))

In [ ]:
import spatialdata as sd
p7513_zarr = sd.read_zarr(REPO_ROOT / "results" / "P7513" / "merscope" / "latest" / "latest_spatialdata.zarr")

In [ ]:
#plt.hist(p7513_zarr.tables["table_MOSAIK_proseg"].X.sum(axis=1), bins=50, range=(0, 500))
np.median(p7513_zarr.tables["table_MOSAIK_proseg"].X.sum(axis=0).flatten().tolist()[0])

In [ ]:
p7513_zarr.tables

In [ ]:
def matrix_row_sums(matrix) -> np.ndarray:
    """Return per-cell transcript counts from an AnnData X matrix."""
    return np.asarray(matrix.sum(axis=1)).ravel()


def matrix_nonzero_by_row(matrix) -> np.ndarray:
    """Return per-cell detected-gene counts from an AnnData X matrix."""
    if sparse.issparse(matrix):
        return np.diff(matrix.tocsr().indptr)
    return np.asarray((matrix > 0).sum(axis=1)).ravel()


_SHAPE_AREA_LOOKUP_CACHE: dict[tuple[str, str], dict[str, pd.Series]] = {}


def _deduplicated_area_lookup(values, areas: np.ndarray) -> pd.Series:
    lookup = pd.Series(areas, index=pd.Index(values).astype(str))
    return lookup[~lookup.index.duplicated(keep="first")]


def shape_area_lookups(zarr_path: Path, region: str) -> dict[str, pd.Series]:
    """Return cached polygon-area lookup tables for one SpatialData shape."""
    cache_key = (str(zarr_path.resolve()), str(region))
    if cache_key in _SHAPE_AREA_LOOKUP_CACHE:
        return _SHAPE_AREA_LOOKUP_CACHE[cache_key]

    print(f"  Loading shape areas for region {region!r} from {zarr_path.name}")
    sdata = sd.read_zarr(zarr_path)
    if region not in sdata.shapes:
        raise KeyError(f"Shape region {region!r} not found in {zarr_path}")

    shapes = sdata.shapes[region]
    areas = np.asarray(shapes.geometry.area, dtype=float)
    lookups = {
        "__index__": _deduplicated_area_lookup(shapes.index, areas),
        "__row_order__": pd.Series(areas),
    }

    for col in shapes.columns:
        if col == "geometry":
            continue
        values = shapes[col]
        if values.notna().any():
            lookups[str(col)] = _deduplicated_area_lookup(values, areas)

    _SHAPE_AREA_LOOKUP_CACHE[cache_key] = lookups
    return lookups


def _obs_values_for_lookup(adata: ad.AnnData, obs_key: str) -> pd.Series:
    if obs_key == "__obs_names__":
        return pd.Series(adata.obs_names.astype(str), index=adata.obs_names)
    return adata.obs[obs_key].astype(str)


def cell_area_um2_for_table(zarr_path: Path, adata: ad.AnnData) -> np.ndarray:
    """Resolve per-cell 2D areas from the table-linked SpatialData shape."""
    attrs = dict(adata.uns.get("spatialdata_attrs", {}))
    region = attrs.get("region")
    instance_key = attrs.get("instance_key")

    if region is not None:
        lookups = shape_area_lookups(zarr_path, str(region))
        candidate_pairs = []
        if instance_key in adata.obs:
            if instance_key in lookups:
                candidate_pairs.append((instance_key, instance_key))
            candidate_pairs.append((instance_key, "__index__"))

        for obs_col in ("cell_id", "EntityID", "cell", "cell_labels"):
            if obs_col not in adata.obs:
                continue
            if obs_col in lookups:
                candidate_pairs.append((obs_col, obs_col))
            candidate_pairs.append((obs_col, "__index__"))

        candidate_pairs.append(("__obs_names__", "__index__"))

        best_areas = None
        best_n = -1
        best_desc = None
        for obs_key, lookup_key in candidate_pairs:
            if lookup_key not in lookups:
                continue
            areas = pd.to_numeric(
                _obs_values_for_lookup(adata, obs_key).map(lookups[lookup_key]),
                errors="coerce",
            ).to_numpy(dtype=float)
            n_mapped = int(np.isfinite(areas).sum())
            if n_mapped > best_n:
                best_areas = areas
                best_n = n_mapped
                best_desc = f"obs[{obs_key!r}] -> shape[{lookup_key!r}]"
            if n_mapped == adata.n_obs:
                return areas

        if best_areas is not None and best_n >= int(0.95 * adata.n_obs):
            print(f"Using partial area mapping for {region}: {best_desc} ({best_n}/{adata.n_obs})")
            return best_areas

        row_order_areas = lookups["__row_order__"].to_numpy(dtype=float)
        if len(row_order_areas) == adata.n_obs:
            print(f"Using row-order area mapping for {region}")
            return row_order_areas

    for col in ("cell_area_um2", "cell_area", "area_um2", "area", "volume"):
        if col in adata.obs:
            areas = pd.to_numeric(adata.obs[col], errors="coerce").to_numpy(dtype=float)
            if np.isfinite(areas).any():
                print(f"Using obs[{col!r}] as cell area")
                return areas

    raise KeyError("Could not resolve cell areas from table metadata, shapes, or obs columns.")


def area_keep_mask(cell_area_um2: np.ndarray) -> np.ndarray:
    keep = np.isfinite(cell_area_um2)
    if MIN_CELL_AREA_UM2 is not None:
        keep &= cell_area_um2 >= float(MIN_CELL_AREA_UM2)
    if MAX_CELL_AREA_UM2 is not None:
        keep &= cell_area_um2 <= float(MAX_CELL_AREA_UM2)
    return keep


def table_cell_ids(adata: ad.AnnData) -> np.ndarray:
    attrs = dict(adata.uns.get("spatialdata_attrs", {}))
    for col in ("cell_id", attrs.get("instance_key"), "EntityID", "cell", "cell_labels"):
        if col in adata.obs:
            return adata.obs[col].astype(str).to_numpy()
    return adata.obs_names.astype(str)


def compute_table_qc(
    zarr_path: Path,
    table_key: str,
    *,
    progress_label: str = "",
) -> tuple[dict[str, object], pd.DataFrame]:
    """Load one AnnData table and compute filtered summaries plus per-cell distributions."""
    prefix = f"[{progress_label}] " if progress_label else ""
    table_path = zarr_path / "tables" / table_key
    if not table_path.exists():
        raise FileNotFoundError(f"Missing table {table_key!r} in {zarr_path}")

    print(f"{prefix}Loading table {table_key!r}")
    adata = ad.read_zarr(table_path)
    print(f"{prefix}Computing genes/transcripts per cell for {adata.n_obs:,} cells")
    transcript_counts = matrix_row_sums(adata.X)
    gene_counts = matrix_nonzero_by_row(adata.X)
    print(f"{prefix}Resolving cell areas from linked SpatialData shapes")
    cell_area_um2 = cell_area_um2_for_table(zarr_path, adata)

    print(f"{prefix}Applying count and {MIN_CELL_AREA_UM2}-{MAX_CELL_AREA_UM2} um2 area filters")
    count_keep = (transcript_counts > MIN_COUNT_EXCLUSIVE) & (gene_counts > MIN_COUNT_EXCLUSIVE)
    area_keep = area_keep_mask(cell_area_um2)
    keep_cells = count_keep & area_keep
    if not np.any(keep_cells):
        raise ValueError(
            f"No cells passed the count and {MIN_CELL_AREA_UM2}-{MAX_CELL_AREA_UM2} um2 area filters for {table_path}"
        )

    print(f"{prefix}Computing median and mean summaries from {int(keep_cells.sum()):,} kept cells")
    metrics = {
        "table_key": table_key,
        "n_cells": int(adata.n_obs),
        "n_cells_after_filter": int(keep_cells.sum()),
        "n_cells_removed": int((~keep_cells).sum()),
        "n_cells_after_count_filter": int(count_keep.sum()),
        "n_cells_removed_by_count_filter": int((~count_keep).sum()),
        "n_cells_after_area_filter": int(area_keep.sum()),
        "n_cells_removed_by_area_filter": int((~area_keep).sum()),
        "min_cell_area_um2": MIN_CELL_AREA_UM2,
        "max_cell_area_um2": MAX_CELL_AREA_UM2,
        "median_cell_area_um2_after_filter": float(np.median(cell_area_um2[keep_cells])),
        "median_genes_per_cell": float(np.median(gene_counts[keep_cells])),
        "median_transcripts_per_cell": float(np.median(transcript_counts[keep_cells])),
        "mean_genes_per_cell": float(np.mean(gene_counts[keep_cells])),
        "mean_transcripts_per_cell": float(np.mean(transcript_counts[keep_cells])),
    }

    print(f"{prefix}Capturing filtered per-cell distributions")
    distributions = pd.DataFrame({
        "cell_id": table_cell_ids(adata)[keep_cells],
        "genes_per_cell": gene_counts[keep_cells].astype(np.int32, copy=False),
        "transcripts_per_cell": transcript_counts[keep_cells].astype(np.float32, copy=False),
        "cell_area_um2": cell_area_um2[keep_cells].astype(np.float32, copy=False),
    })
    return metrics, distributions


def compute_all_qc(spatialdata_outputs: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    tasks = []
    for output in spatialdata_outputs.itertuples(index=False):
        for segmentation, style in SEGMENTATION_STYLES.items():
            table_path = output.zarr_path / "tables" / style["table_key"]
            if table_path.exists():
                tasks.append((output, segmentation, style))

    if not tasks:
        raise FileNotFoundError("No requested AnnData tables were found in the SpatialData outputs.")

    print(f"Computing QC summaries and per-cell distributions for {len(tasks)} tables")
    rows = []
    distribution_rows = []
    for output, segmentation, style in tqdm(tasks, desc="Computing medians/means", unit="table"):
        label = f"{output.pair_id} {output.platform} {segmentation}"
        print(f"\nStarting {label}")
        metrics, distributions = compute_table_qc(
            output.zarr_path,
            style["table_key"],
            progress_label=label,
        )
        rows.append({
            "pair_id": output.pair_id,
            "platform": output.platform,
            "segmentation": segmentation,
            "zarr_path": str(output.zarr_path.relative_to(REPO_ROOT)),
            **metrics,
        })
        distributions.insert(0, "zarr_path", str(output.zarr_path.relative_to(REPO_ROOT)))
        distributions.insert(0, "table_key", style["table_key"])
        distributions.insert(0, "segmentation", segmentation)
        distributions.insert(0, "platform", output.platform)
        distributions.insert(0, "pair_id", output.pair_id)
        distribution_rows.append(distributions)
        print(f"Finished {label}: kept {metrics['n_cells_after_filter']:,}/{metrics['n_cells']:,} cells")

    print("Combining summary metrics and per-cell distributions")
    metrics = pd.DataFrame(rows).sort_values(["pair_id", "segmentation", "platform"]).reset_index(drop=True)
    distributions = pd.concat(distribution_rows, ignore_index=True)
    distributions = distributions.sort_values(["segmentation", "platform", "pair_id", "cell_id"]).reset_index(drop=True)
    return metrics, distributions


metrics, cell_distributions = compute_all_qc(spatialdata_outputs)
print(f"Saving filtered per-cell distributions to {DISTRIBUTION_PATH}")
cell_distributions.to_csv(DISTRIBUTION_PATH, index=False, compression="gzip")
display(metrics)
print(f"Saved filtered per-cell distributions to {DISTRIBUTION_PATH}")

In [ ]:
def sample_jitter(metrics: pd.DataFrame, width: float = 0.055) -> dict[str, float]:
    """Small deterministic x offsets so overlapping sample points remain visible."""
    sample_ids = sorted(metrics["pair_id"].unique())
    if len(sample_ids) == 1:
        return {sample_ids[0]: 0.0}
    return dict(zip(sample_ids, np.linspace(-width, width, len(sample_ids)), strict=True))


JITTER = sample_jitter(metrics)


def x_position(pair_id: str, platform: str, segmentation: str) -> float:
    return PLATFORM_X[platform] + SEGMENTATION_STYLES[segmentation]["offset"] + JITTER[pair_id]


def plot_metric_panel(ax: plt.Axes, metric_column: str, ylabel: str, title: str) -> None:
    for segmentation, style in SEGMENTATION_STYLES.items():
        seg_df = metrics.loc[metrics["segmentation"] == segmentation].copy()

        for pair_id, sample_df in seg_df.groupby("pair_id", sort=True):
            platform_df = sample_df.set_index("platform")
            if {"merscope", "xenium"}.issubset(platform_df.index):
                xs = [
                    x_position(pair_id, "merscope", segmentation),
                    x_position(pair_id, "xenium", segmentation),
                ]
                ys = [
                    platform_df.loc["merscope", metric_column],
                    platform_df.loc["xenium", metric_column],
                ]
                ax.plot(xs, ys, color=style["color"], linewidth=1.2, alpha=0.45, zorder=1)

        xs = [x_position(row.pair_id, row.platform, row.segmentation) for row in seg_df.itertuples()]
        ax.scatter(
            xs,
            seg_df[metric_column],
            s=58,
            marker="o",
            color=style["color"],
            edgecolors="white",
            linewidths=0.8,
            alpha=0.95,
            zorder=3,
        )

    ax.set_xticks([PLATFORM_X["merscope"], PLATFORM_X["xenium"]])
    ax.set_xticklabels([PLATFORM_LABELS["merscope"], PLATFORM_LABELS["xenium"]])
    ax.set_xlim(-0.22, 1.22)
    ax.set_ylim(bottom=0)
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.grid(axis="y", color="0.88", linewidth=0.8)
    ax.tick_params(axis="x", length=0)


print("Plotting median and mean summary panels")
fig, axes = plt.subplots(2, 2, figsize=(11, 8.2), sharex=True)

plot_metric_panel(axes[0, 0], "median_genes_per_cell", "Genes per cell", "Median genes")
plot_metric_panel(axes[0, 1], "median_transcripts_per_cell", "Transcripts per cell", "Median transcripts")
plot_metric_panel(axes[1, 0], "mean_genes_per_cell", "Genes per cell", "Mean genes")
plot_metric_panel(axes[1, 1], "mean_transcripts_per_cell", "Transcripts per cell", "Mean transcripts")

legend_handles = [
    plt.Line2D(
        [0],
        [0],
        marker="o",
        color=style["color"],
        markerfacecolor=style["color"],
        markeredgecolor="white",
        markersize=7,
        linewidth=1.4,
        label=style["label"],
    )
    for style in SEGMENTATION_STYLES.values()
]
fig.legend(handles=legend_handles, loc="upper center", ncol=2, frameon=False, bbox_to_anchor=(0.5, 1.02))
fig.tight_layout(rect=(0, 0, 1, 0.9))

print(f"Saving summary figure to {SUMMARY_FIGURE_PATH}")
fig.savefig(SUMMARY_FIGURE_PATH, bbox_inches="tight")
display(SUMMARY_FIGURE_PATH)


PLATFORM_COLORS = {"merscope": "#1f77b4", "xenium": "#2ca02c"}


def letter_label(i: int) -> str:
    alphabet = "ABCDEFGHIJKLMNOPQRSTUVWXYZ"
    return alphabet[i] if i < len(alphabet) else f"S{i + 1}"


def platform_sample_aliases(distributions: pd.DataFrame) -> dict[str, dict[str, str]]:
    aliases = {}
    for platform in PLATFORM_X:
        pair_ids = sorted(distributions.loc[distributions["platform"] == platform, "pair_id"].unique())
        if DEIDENTIFY_SAMPLE_LABELS:
            aliases[platform] = {pair_id: letter_label(i) for i, pair_id in enumerate(pair_ids)}
        else:
            aliases[platform] = {pair_id: pair_id for pair_id in pair_ids}
    return aliases


VIOLIN_SAMPLE_ALIASES = platform_sample_aliases(cell_distributions)


def values_for_violin(values: pd.Series, seed_offset: int) -> np.ndarray:
    arr = values.to_numpy(dtype=float)
    arr = arr[np.isfinite(arr)]
    if VIOLIN_MAX_CELLS_PER_GROUP is not None and len(arr) > VIOLIN_MAX_CELLS_PER_GROUP:
        rng = np.random.default_rng(int(VIOLIN_RANDOM_SEED) + int(seed_offset))
        arr = rng.choice(arr, size=int(VIOLIN_MAX_CELLS_PER_GROUP), replace=False)
    return arr


def violin_groups(distributions: pd.DataFrame, segmentation: str, metric_column: str):
    seg_df = distributions.loc[distributions["segmentation"] == segmentation]
    data = []
    positions = []
    labels = []
    colors = []
    platform_centers = {}
    pos = 0.0

    for platform in PLATFORM_X:
        start_pos = pos
        platform_df = seg_df.loc[seg_df["platform"] == platform]
        for j, pair_id in enumerate(sorted(platform_df["pair_id"].unique())):
            group = platform_df.loc[platform_df["pair_id"] == pair_id, metric_column]
            values = values_for_violin(group, seed_offset=len(data) + 1)
            if len(values) < 2:
                continue
            data.append(values)
            positions.append(pos)
            labels.append(VIOLIN_SAMPLE_ALIASES[platform].get(pair_id, pair_id))
            colors.append(PLATFORM_COLORS[platform])
            pos += 1.0
        if pos > start_pos:
            platform_centers[platform] = (start_pos + pos - 1.0) / 2.0
            pos += 1.7

    return data, positions, labels, colors, platform_centers


def plot_violin_panel(ax: plt.Axes, segmentation: str, metric_column: str, ylabel: str) -> None:
    data, positions, labels, colors, platform_centers = violin_groups(
        cell_distributions,
        segmentation,
        metric_column,
    )
    parts = ax.violinplot(data, positions=positions, widths=0.82, showmedians=True, showextrema=False)
    for body, color in zip(parts["bodies"], colors, strict=True):
        body.set_facecolor(color)
        body.set_edgecolor("black")
        body.set_alpha(0.55)
        body.set_linewidth(0.6)
    if "cmedians" in parts:
        parts["cmedians"].set_color("black")
        parts["cmedians"].set_linewidth(1.0)

    ax.set_xticks(positions)
    ax.set_xticklabels(labels, rotation=45, ha="right")
    for platform, center in platform_centers.items():
        ax.text(center, -0.22, PLATFORM_LABELS[platform], transform=ax.get_xaxis_transform(), ha="center", va="top", fontweight="bold")
    ax.set_xlim(min(positions) - 0.8, max(positions) + 0.8)
    if metric_column == "transcripts_per_cell":
        positive_min = min(float(np.nanmin(values)) for values in data if len(values))
        ax.set_yscale("log")
        ax.set_ylim(bottom=max(1.0, positive_min * 0.8))
    else:
        ax.set_ylim(bottom=0)
    ax.set_ylabel(ylabel)
    ax.grid(axis="y", color="0.9", linewidth=0.8)


print("Plotting per-cell distribution violins")
fig, axes = plt.subplots(2, 2, figsize=(14, 8.8), sharex=False)
for row, segmentation in enumerate(["original_seg", "reseg"]):
    print(f"  Plotting violins for {SEGMENTATION_STYLES[segmentation]['label']}")
    plot_violin_panel(axes[row, 0], segmentation, "genes_per_cell", "Genes per cell")
    plot_violin_panel(axes[row, 1], segmentation, "transcripts_per_cell", "Transcripts per cell")
    axes[row, 0].set_title(SEGMENTATION_STYLES[segmentation]["label"])
    axes[row, 1].set_title(SEGMENTATION_STYLES[segmentation]["label"])

fig.tight_layout(h_pad=2.8)
print(f"Saving violin figure to {VIOLIN_FIGURE_PATH}")
fig.savefig(VIOLIN_FIGURE_PATH, bbox_inches="tight")
VIOLIN_FIGURE_PATH